# UAV 5G Co-Simulation — document workflow

This is the single implementation notebook for `UAV_5GSim`. It follows the supplied student document in order. It does not import or execute any EiraX `.py` file. Labels may come from a scenario manifest; measurements and behavioral flags must come from observed behavior.

Run one numbered section at a time. Do not continue past a failed checkpoint.

## 0. Clean previous UAV notebook session

Run this cell first. It stops only processes belonging to this UAV notebook, releases their memory, and reports available RAM. It does not delete datasets or drop the Linux filesystem cache.

In [1]:
import gc, os, signal, subprocess, time
from pathlib import Path

def available_ram_gib():
    values={}
    for line in Path('/proc/meminfo').read_text().splitlines():
        key,value=line.split(':',1); values[key]=int(value.strip().split()[0])
    return values['MemAvailable']/1024/1024

def process_ancestors(pid):
    result=set()
    while pid>1 and pid not in result:
        result.add(pid)
        try:
            fields=Path(f'/proc/{pid}/stat').read_text().split(); pid=int(fields[3])
        except (FileNotFoundError,IndexError,ValueError): break
    return result

def clean_previous_uav_session():
    before=available_ram_gib(); protected=process_ancestors(os.getpid())
    markers=(
      '/UAV_5GSim/worlds/baylands_uav_goal.world',
      '/UGV_UAV/px4_runtime/bin/px4',
      'fcu_url:=udp://:14550@127.0.0.1:18570',
      'ros2 bag record -o /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/',
    )
    listing=subprocess.run(['ps','-eo','pid=,args='],capture_output=True,text=True,check=True).stdout.splitlines()
    groups={}
    for line in listing:
        fields=line.strip().split(None,1)
        if len(fields)!=2: continue
        pid=int(fields[0]); command=fields[1]
        if pid in protected or not any(marker in command for marker in markers): continue
        try: groups.setdefault(os.getpgid(pid),command)
        except ProcessLookupError: pass
    for pgid in groups:
        try: os.killpg(pgid,signal.SIGINT)
        except ProcessLookupError: pass
    if groups: time.sleep(3)
    for pgid in groups:
        try: os.killpg(pgid,signal.SIGTERM)
        except ProcessLookupError: pass
    gc.collect(); time.sleep(.5); after=available_ram_gib()
    if groups:
        print(f'[CLEANED] Closed {len(groups)} previous UAV process group(s).')
        for command in groups.values(): print(' -',command[:140])
    else: print('[CLEAN] No previous UAV notebook process is running.')
    print(f'[RAM] Available before={before:.1f} GiB, after={after:.1f} GiB')

clean_previous_uav_session()

[CLEAN] No previous UAV notebook process is running.
[RAM] Available before=8.8 GiB, after=8.8 GiB


## 1–3. Configuration and environment checkpoint

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import csv, json, math, os, shutil, signal, subprocess, sys, threading, time

HERE = Path.cwd().resolve()
assert HERE.name == 'UAV_5GSim', f'Open the notebook from UAV_5GSim, not {HERE}'
PROJECT = HERE.parent
WORLD = HERE / 'worlds/baylands_uav_goal.world'
GUI_CONFIG = HERE / 'uav_gui.config'
PX4 = PROJECT / 'UGV_UAV/px4_runtime'
MODELS = PROJECT / 'simulation/models'
SIMU5G = PROJECT / 'UGV_UAV_5G_CoSimulation/sim/simu5g/Simu5G'
OMNETPP = Path('/home/basudeo/omnetpp-6.0.1')
SCHEMA = HERE / 'schema_reference.json'

WORLD_NAME = 'baylands_editable'
UAV_MODEL = 'gz_x500_mapping'
UAV_ENTITY = 'x500_mapping_0'
SPAWN = {'x': 211.12966918945318, 'y': -256.3474426269531, 'z': 2.935702896118163}
GOAL = {'x': -29.546520233154297, 'y': -286.43212890625, 'z': 1.6005200147628784}
GOAL_NORTH = GOAL['y'] - SPAWN['y']
GOAL_EAST = GOAL['x'] - SPAWN['x']
GOAL_GROUND_DOWN = SPAWN['z'] - GOAL['z']
ALTITUDE_M = 30.0
SPEED_MPS = 5.0

checks = {
    'Ubuntu 24.04': '24.04' in Path('/etc/os-release').read_text(),
    'ROS 2': Path('/opt/ros/jazzy').is_dir(),
    'Gazebo': shutil.which('gz') is not None,
    'PX4 runtime': (PX4 / 'bin/px4').is_file(),
    'PX4 x500 base model': (PX4 / 'models/x500_base/model.sdf').is_file(),
    'PX4 Gazebo plugins': (PX4 / 'plugins/libMotorFailurePlugin.so').is_file(),
    'PX4 Python MAVLink': (PX4 / 'python/pymavlink').is_dir(),
    'custom x500 mapping drone': (MODELS / 'x500_mapping/model.sdf').is_file(),
    'custom x500 mapping base': (MODELS / 'x500_mapping_base/model.sdf').is_file(),
    '32-channel UAV LiDAR': 'sensor name="lidar_32"' in (MODELS / 'x500_mapping_base/model.sdf').read_text(),
    'UAV loaded in isolated world': '<name>x500_mapping_0</name>' in WORLD.read_text(),
    'MAVROS': Path('/opt/ros/jazzy/share/mavros').is_dir(),
    'MAVROS messages': Path('/opt/ros/jazzy/share/mavros_msgs').is_dir(),
    'MAVROS GeographicLib geoid': Path('/usr/share/GeographicLib/geoids/egm96-5.pgm').is_file(),
    'isolated world': WORLD.is_file(),
    'UAV-centered Gazebo view': GUI_CONFIG.is_file(),
    'OMNeT++': OMNETPP.is_dir(),
    'Simu5G': SIMU5G.is_dir(),
}
for name, passed in checks.items(): print('[OK]' if passed else '[MISSING]', name)
missing=[name for name,passed in checks.items() if not passed]
if 'MAVROS GeographicLib geoid' in missing:
    print('[FIX] Run in a terminal: sudo geographiclib-get-geoids egm96-5')
assert not missing, 'Environment checkpoint failed; missing: '+', '.join(missing)
print(f'[MISSION] distance={math.hypot(GOAL_NORTH, GOAL_EAST):.1f} m')

[OK] Ubuntu 24.04
[OK] ROS 2
[OK] Gazebo
[OK] PX4 runtime
[OK] PX4 x500 base model
[OK] PX4 Gazebo plugins
[OK] PX4 Python MAVLink
[OK] custom x500 mapping drone
[OK] custom x500 mapping base
[OK] 32-channel UAV LiDAR
[OK] UAV loaded in isolated world
[OK] MAVROS
[OK] MAVROS messages
[OK] MAVROS GeographicLib geoid
[OK] isolated world
[OK] UAV-centered Gazebo view
[OK] OMNeT++
[OK] Simu5G
[MISSION] distance=242.5 m


## 4. Start Gazebo, PX4, MAVROS and the benign ROS 2 bag

Gazebo starts paused. This cell creates a new run directory and starts recording the three document-required MAVROS topics.

In [3]:
PROCESSES = {}
RUN = None
LOG_HANDLES = []

def stop_process(process):
    if process is None or process.poll() is not None: return
    os.killpg(process.pid, signal.SIGINT)
    try: process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait(timeout=5)

def start_logged(name, command, env, cwd=HERE):
    log = (RUN / f'{name}.log').open('w', encoding='utf-8')
    LOG_HANDLES.append(log)
    process = subprocess.Popen(command, cwd=cwd, env=env, stdin=subprocess.PIPE if name == 'px4' else subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    PROCESSES[name] = process
    return process

def follow_uav_in_gazebo():
    requests=[
      ('/gui/follow/offset','gz.msgs.Vector3d','x: -5.0, y: -5.0, z: 3.0'),
      ('/gui/follow','gz.msgs.StringMsg',f'data: "{UAV_ENTITY}"'),
    ]
    for service,request_type,request in requests:
        accepted=False
        for _ in range(10):
            result=subprocess.run(['gz','service','-s',service,'--reqtype',request_type,'--reptype','gz.msgs.Boolean','--timeout','2000','--req',request],capture_output=True,text=True,timeout=5)
            if result.returncode==0 and 'data: true' in result.stdout:
                accepted=True; break
            time.sleep(1)
        assert accepted, f'Gazebo camera service failed: {service}'
    print(f'[CAMERA] Following {UAV_ENTITY}; the UAV should now be centered in Gazebo.')

def start_benign_run():
    global RUN
    services = subprocess.run(['gz', 'service', '-l'], capture_output=True, text=True, timeout=10)
    assert '/server_control' not in services.stdout.splitlines(), 'Close the existing Gazebo server first'
    RUN = HERE / 'notebook_runs' / datetime.now().strftime('run_%Y%m%d_%H%M%S')
    RUN.mkdir(parents=True)
    env = os.environ.copy()
    env['QT_QPA_PLATFORM'] = env.get('QT_QPA_PLATFORM', 'xcb')
    env['GZ_SIM_RESOURCE_PATH'] = ':'.join([str(MODELS), str(PX4 / 'models'), env.get('GZ_SIM_RESOURCE_PATH', '')]).rstrip(':')
    env['GZ_SIM_SYSTEM_PLUGIN_PATH'] = ':'.join([str(PX4 / 'plugins'), env.get('GZ_SIM_SYSTEM_PLUGIN_PATH', '')]).rstrip(':')
    start_logged('gazebo', ['gz', 'sim', '--gui-config', str(GUI_CONFIG), str(WORLD)], env)
    print('[PAUSED] Baylands is loading. Do not click Play yet.')
    time.sleep(6)
    assert PROCESSES['gazebo'].poll() is None, f'Gazebo failed; see {RUN / "gazebo.log"}'
    px4_env = env.copy()
    px4_env.pop('PX4_SIM_MODEL',None); px4_env.pop('PX4_GZ_MODEL_POSE',None)
    px4_env.update({'PX4_SYS_AUTOSTART':'4001', 'PX4_GZ_MODEL_NAME':UAV_ENTITY, 'PX4_GZ_STANDALONE':'1', 'PX4_GZ_NO_FOLLOW':'1', 'PX4_GZ_WORLD':WORLD_NAME})
    start_logged('px4', [str(PX4 / 'bin/px4'), '-d'], px4_env, PX4 / 'rootfs')
    time.sleep(8)
    assert PROCESSES['px4'].poll() is None, f'PX4 failed; see {RUN / "px4.log"}'
    follow_uav_in_gazebo()
    ros_prefix = 'source /opt/ros/jazzy/setup.bash && '
    mavros = ros_prefix + 'exec ros2 run mavros mavros_node --ros-args -p fcu_url:=udp://:14550@127.0.0.1:18570'
    start_logged('mavros', ['bash', '-lc', mavros], env)
    time.sleep(4)
    assert PROCESSES['mavros'].poll() is None, f'MAVROS failed; see {RUN / "mavros.log"}'
    bag_dir = RUN / 'rosbag_benign'
    bag = ros_prefix + f'exec ros2 bag record -o {bag_dir} /mavros/local_position/pose /mavros/state /mavros/global_position/raw/fix'
    start_logged('rosbag', ['bash', '-lc', bag], env)
    print('[READY] ROS bag recording started:', bag_dir)
    print('[ACTION] Now click Play in Gazebo, then run the next cell.')
    return RUN

start_benign_run()

[PAUSED] Baylands is loading. Do not click Play yet.
[CAMERA] Following x500_mapping_0; the UAV should now be centered in Gazebo.
[READY] ROS bag recording started: /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/rosbag_benign
[ACTION] Now click Play in Gazebo, then run the next cell.


PosixPath('/home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640')

## 4. Fly spawn → goal → land

The complete PX4 command logic is defined here. It does not call another Python file.

In [4]:
sys.path.insert(0, str(PX4 / 'python'))
from pymavlink import mavutil

MASK = (mavutil.mavlink.POSITION_TARGET_TYPEMASK_VX_IGNORE | mavutil.mavlink.POSITION_TARGET_TYPEMASK_VY_IGNORE | mavutil.mavlink.POSITION_TARGET_TYPEMASK_VZ_IGNORE | mavutil.mavlink.POSITION_TARGET_TYPEMASK_AX_IGNORE | mavutil.mavlink.POSITION_TARGET_TYPEMASK_AY_IGNORE | mavutil.mavlink.POSITION_TARGET_TYPEMASK_AZ_IGNORE | mavutil.mavlink.POSITION_TARGET_TYPEMASK_YAW_IGNORE | mavutil.mavlink.POSITION_TARGET_TYPEMASK_YAW_RATE_IGNORE)

def setpoint(c, n, e, d):
    c.mav.set_position_target_local_ned_send(int(time.monotonic()*1000)&0xffffffff, c.target_system, c.target_component, mavutil.mavlink.MAV_FRAME_LOCAL_NED, MASK, n,e,d, 0,0,0, 0,0,0, 0,0)

def command(c, number, *params, timeout=10):
    values = list(params) + [0.0] * (7-len(params))
    c.mav.command_long_send(c.target_system, c.target_component, number, 0, *values[:7])
    end = time.monotonic() + timeout
    while time.monotonic() < end:
        ack = c.recv_match(type='COMMAND_ACK', blocking=True, timeout=1)
        if ack and ack.command == number:
            if ack.result in (mavutil.mavlink.MAV_RESULT_ACCEPTED, mavutil.mavlink.MAV_RESULT_IN_PROGRESS): return
            raise RuntimeError(f'PX4 rejected command {number}: {ack.result}')
    raise TimeoutError(f'No acknowledgement for command {number}')

def fly_benign():
    c = mavutil.mavlink_connection('udpin:0.0.0.0:14540', source_system=250)
    assert c.wait_heartbeat(timeout=30), 'No PX4 heartbeat'
    stop_hb = threading.Event(); stop_sp = threading.Event(); active = [0.0, 0.0, -ALTITUDE_M]
    def heartbeat():
        while not stop_hb.is_set():
            c.mav.heartbeat_send(mavutil.mavlink.MAV_TYPE_GCS, mavutil.mavlink.MAV_AUTOPILOT_INVALID, 0,0,mavutil.mavlink.MAV_STATE_ACTIVE); stop_hb.wait(.8)
    def stream():
        while not stop_sp.is_set(): setpoint(c, *active); stop_sp.wait(.1)
    hb = threading.Thread(target=heartbeat, daemon=True); hb.start()
    try:
        print('[WAITING] Position and heading...')
        end = time.monotonic()+300
        while time.monotonic()<end:
            p=c.recv_match(type='GLOBAL_POSITION_INT', blocking=True, timeout=1)
            if p and p.lat and p.lon: break
        else: raise TimeoutError('No valid PX4 position')
        for name in ('MPC_XY_VEL_MAX','MPC_XY_CRUISE'):
            c.mav.param_set_send(c.target_system,c.target_component,name.encode(),SPEED_MPS,mavutil.mavlink.MAV_PARAM_TYPE_REAL32)
            limit=time.monotonic()+5
            while time.monotonic()<limit:
                v=c.recv_match(type='PARAM_VALUE',blocking=True,timeout=1)
                if v and v.param_id.rstrip('\x00')==name: break
        sp=threading.Thread(target=stream,daemon=True); sp.start(); time.sleep(2)
        arm_end=time.monotonic()+45
        while True:
            try: command(c,mavutil.mavlink.MAV_CMD_COMPONENT_ARM_DISARM,1.0,timeout=5); break
            except RuntimeError:
                if time.monotonic()>arm_end: raise
                time.sleep(2)
        c.mav.set_mode_send(c.target_system,mavutil.mavlink.MAV_MODE_FLAG_CUSTOM_MODE_ENABLED,6<<16)
        targets=[('takeoff',0,0,-ALTITUDE_M,1.5),('goal',GOAL_NORTH,GOAL_EAST,-ALTITUDE_M,3.0),('approach',GOAL_NORTH,GOAL_EAST,GOAL_GROUND_DOWN-2.0,1.0)]
        trace=RUN/'pose_trace.csv'; start=time.monotonic()
        with trace.open('w',newline='') as f:
            w=csv.writer(f); w.writerow(['timestamp_sec','phase','north_m','east_m','down_m'])
            for phase,n,e,d,radius in targets:
                active[:]=[n,e,d]
                while time.monotonic()-start<300:
                    p=c.recv_match(type='LOCAL_POSITION_NED',blocking=True,timeout=.2)
                    if not p: continue
                    w.writerow([f'{time.monotonic()-start:.3f}',phase,p.x,p.y,p.z]); f.flush()
                    error=math.sqrt((p.x-n)**2+(p.y-e)**2+(p.z-d)**2)
                    if error<=radius: print('[REACHED]',phase,f'{error:.2f} m'); break
                else: raise TimeoutError(f'{phase} timed out')
        command(c,mavutil.mavlink.MAV_CMD_NAV_LAND)
        end=time.monotonic()+120
        while time.monotonic()<end:
            h=c.recv_match(type='HEARTBEAT',blocking=True,timeout=1)
            if h and not (h.base_mode & mavutil.mavlink.MAV_MODE_FLAG_SAFETY_ARMED):
                print('[LANDED] UAV landed and disarmed at the goal'); return trace
        raise TimeoutError('Landing was not confirmed')
    finally:
        stop_sp.set(); stop_hb.set(); c.close()

POSE_TRACE = fly_benign()
POSE_TRACE

[WAITING] Position and heading...
[REACHED] takeoff 1.46 m
[REACHED] goal 2.93 m
[REACHED] approach 0.98 m
[LANDED] UAV landed and disarmed at the goal


PosixPath('/home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/pose_trace.csv')

## 4. Stop recording and verify the benign checkpoint

In [5]:
stop_process(PROCESSES.get('rosbag'))
bag = RUN / 'rosbag_benign'
info = subprocess.run(['bash','-lc',f'source /opt/ros/jazzy/setup.bash && ros2 bag info {bag}'],capture_output=True,text=True,timeout=30)
print(info.stdout); assert info.returncode == 0, info.stderr
rows = list(csv.DictReader(POSE_TRACE.open()))
assert rows, 'Pose trace is empty'
simu_trace = RUN / 'simu5g_pose_trace.txt'
with simu_trace.open('w') as f:
    f.write('time_sec x_east_m y_north_m z_up_m\n')
    for row in rows:
        f.write(f"{row['timestamp_sec']} {row['east_m']} {row['north_m']} {-float(row['down_m'])}\n")
print('[CHECKPOINT PASSED] ROS bag:', bag)
print('[CHECKPOINT PASSED] Simu5G pose trace:', simu_trace)


Files:             rosbag_benign_0.mcap
Bag size:          24.0 MiB
Storage id:        mcap
ROS Distro:        jazzy
Duration:          528.414486347s
Start:             Aug  5 2026 21:07:00.037039264 (1785956820.037039264)
End:               Aug  5 2026 21:15:48.451525611 (1785957348.451525611)
Messages:          274412
Topic information: Topic: /mavros/global_position/raw/fix | Type: sensor_msgs/msg/NavSatFix | Count: 516 | Serialization Format: cdr
                   Topic: /mavros/local_position/pose | Type: geometry_msgs/msg/PoseStamped | Count: 5125 | Serialization Format: cdr
                   Topic: /mavros/state | Type: mavros_msgs/msg/State | Count: 268771 | Serialization Format: cdr
Service:           0
Service information: 

[CHECKPOINT PASSED] ROS bag: /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/rosbag_benign
[CHECKPOINT PASSED] Simu5G pose trace: /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/simu5g_pose_trace.txt

## 5. One-UAV 5G scenario

This cell generates and runs exactly one UAV, one gNB, one 5G core, one edge node, and one mission backend. The measured Gazebo pose trace becomes BonnMotion mobility. Simu5G produces scalar/vector results and `network_metrics.csv`.

In [6]:
import shlex

NED_TEXT = r'''
import inet.networklayer.configurator.ipv4.Ipv4NetworkConfigurator;
import inet.networklayer.ipv4.RoutingTableRecorder;
import inet.node.ethernet.Eth10G;
import inet.node.inet.Router;
import inet.node.inet.StandardHost;
import simu5g.common.binder.Binder;
import simu5g.common.carrierAggregation.CarrierAggregation;
import simu5g.nodes.Upf;
import simu5g.nodes.NR.gNodeB;
import simu5g.nodes.NR.NRUe;
import simu5g.world.radio.LteChannelControl;
network EiraXOneUas {
 parameters: int numUe = default(1); @display("i=block/network2;bgb=1000,700");
 submodules:
  channelControl: LteChannelControl; routingRecorder: RoutingTableRecorder;
  configurator: Ipv4NetworkConfigurator; binder: Binder; carrierAggregation: CarrierAggregation;
  missionBackend: StandardHost { @display("p=120,100;i=device/server"); }
  edgeNode: StandardHost { @display("p=280,100;i=device/server"); }
  router: Router { @display("p=440,100;i=device/smallrouter"); }
  upf: Upf { @display("p=590,100"); } iUpf: Upf { @display("p=740,100"); }
  gnb: gNodeB { @display("p=740,280;is=vl"); } ue[numUe]: NRUe { @display("p=740,470"); }
 connections:
  missionBackend.pppg++ <--> Eth10G <--> router.pppg++;
  edgeNode.pppg++ <--> Eth10G <--> router.pppg++;
  router.pppg++ <--> Eth10G <--> upf.filterGate;
  upf.pppg++ <--> Eth10G <--> iUpf.pppg++;
  iUpf.pppg++ <--> Eth10G <--> gnb.ppp;
}
'''

SCENARIOS = {
 'baseline': dict(attack_type='none', attack_stage='none', severity='none', incident_label='normal'),
 'suspicious': dict(attack_type='unknown_anomaly', attack_stage='initial_access', severity='low', incident_label='suspicious', activity='authentication', rate=15, size=180, source_ip='10.0.0.21', destination_ip='10.0.0.2', authentication_status='unknown', session_id='auth-retry'),
 'denial_of_service': dict(attack_type='denial_of_service', attack_stage='impact', severity='critical', incident_label='malicious', activity='telemetry', rate=200, size=1200, source_ip='10.0.0.66', destination_ip='10.0.0.2', authentication_status='failed', session_id='flood-1'),
 'session_hijacking': dict(attack_type='session_hijacking', attack_stage='persistence', severity='high', incident_label='malicious', activity='command', rate=10, size=220, source_ip='10.0.0.66', destination_ip='10.0.0.2', authentication_status='success', session_id='command-session'),
 'identity_theft': dict(attack_type='identity_theft', attack_stage='initial_access', severity='high', incident_label='malicious', activity='authentication', rate=20, size=180, source_ip='10.0.0.77', destination_ip='10.0.0.2', authentication_status='failed', session_id='forged-identity'),
 'control_protocol_attack': dict(attack_type='control_protocol_attack', attack_stage='execution', severity='critical', incident_label='malicious', activity='command', rate=30, size=260, source_ip='10.0.0.66', destination_ip='10.0.0.2', authentication_status='success', session_id='rogue-command'),
 'lateral_movement': dict(attack_type='lateral_movement', attack_stage='lateral_movement', severity='high', incident_label='malicious', activity='scan', rate=35, size=128, source_ip='10.0.0.1', destination_ip='10.0.0.99', authentication_status='success', session_id='scan-1'),
 'network_application_attack': dict(attack_type='network_application_attack', attack_stage='impact', severity='high', incident_label='malicious', activity='api', rate=100, size=900, source_ip='10.0.0.66', destination_ip='10.0.0.2', authentication_status='failed', session_id='api-flood'),
}
RESPONSES = {'none':'continue_monitoring','unknown_anomaly':'increase_monitoring','denial_of_service':'rate_limit_and_block_flow','session_hijacking':'revoke_session_force_reauth','identity_theft':'suspend_identity_quarantine_device','control_protocol_attack':'block_command_channel_safe_mode','lateral_movement':'segment_slice_isolate_component','network_application_attack':'rate_limit_api_alert_security_team'}

def measured_flows(scenario):
    flows = [
      dict(name='telemetry', activity='telemetry', rate=20.0, size=732, source_ip='10.0.0.1', destination_ip='10.0.0.2', source_port=7701, destination_port=8701, protocol='UDP', session_id='telemetry-session', authentication_status='success', source_component='drone', destination_component='mission_backend'),
      dict(name='command', activity='command', rate=2.0, size=200, source_ip='10.0.0.1', destination_ip='10.0.0.2', source_port=7702, destination_port=8702, protocol='MAVLink-like', session_id='command-session', authentication_status='success', source_component='drone', destination_component='mission_backend') ]
    cfg = SCENARIOS[scenario]
    if scenario != 'baseline':
        i=len(flows); flows.append(dict(name=scenario, activity=cfg['activity'], rate=float(cfg['rate']), size=int(cfg['size']), source_ip=cfg['source_ip'], destination_ip=cfg['destination_ip'], source_port=7701+i, destination_port=8701+i, protocol='HTTP' if cfg['activity']=='api' else ('MAVLink-like' if cfg['activity']=='command' else 'UDP'), session_id=cfg['session_id'], authentication_status=cfg['authentication_status'], source_component='external_host' if cfg['source_ip']!='10.0.0.1' else 'drone', destination_component='security_monitor' if cfg['activity']=='scan' else 'mission_backend'))
    return flows

def prepare_network(scenario='baseline', seed=0):
    assert scenario in SCENARIOS
    root=RUN/'network'/scenario; inp=root/'input'; raw=root/'raw'; data=root/'data'
    for p in (inp,raw,data): p.mkdir(parents=True,exist_ok=True)
    trace=list(csv.DictReader((RUN/'simu5g_pose_trace.txt').open(),delimiter=' ')); assert trace
    movement=[]
    for r in trace:
        t=float(r['time_sec']); x=500+float(r['x_east_m']); y=500+float(r['y_north_m']); z=max(.05,float(r['z_up_m'])); movement += [f'{t:.3f}',f'{x:.3f}',f'{y:.3f}',f'{z:.3f}']
    (inp/'uav.movements').write_text(' '.join(movement)+'\n')
    duration=max(2.0,float(trace[-1]['time_sec'])+.5); flows=measured_flows(scenario)
    (inp/'EiraXOneUas.ned').write_text(NED_TEXT); (inp/'demo.xml').write_text('<config><interface hosts="*" address="10.x.x.x" netmask="255.255.255.0"/></config>\n')
    ini=['[General]',f'seed-set = {seed}','network = EiraXOneUas',f'sim-time-limit = {duration:.3f}s',f'image-path = {SIMU5G}/images','output-scalar-file = ${resultdir}/uas_benign.sca','output-vector-file = ${resultdir}/uas_benign.vec','output-scalar-file-append = false','**.routingRecorder.enabled = false','**.vector-recording = true','**.mobility.constraintAreaMinX = 0m','**.mobility.constraintAreaMinY = 0m','**.mobility.constraintAreaMinZ = 0m','**.mobility.constraintAreaMaxX = 1000m','**.mobility.constraintAreaMaxY = 1000m','**.mobility.constraintAreaMaxZ = 100m','**.mobility.initFromDisplayString = false','**.numBands = 50','**.ueTxPower = 26','**.eNodeBTxPower = 40','**.targetBler = 0.01','**.blerShift = 5','*.configurator.config = xmldoc("demo.xml")','*.gnb.mobility.initialX = 500m','*.gnb.mobility.initialY = 500m','*.numUe = 1','*.ue[0].macCellId = 0','*.ue[0].masterId = 0','*.ue[0].nrMacCellId = 1','*.ue[0].nrMasterId = 1','*.ue[0].mobility.typename = "BonnMotionMobility"','*.ue[0].mobility.traceFile = "uav.movements"','*.ue[0].mobility.nodeId = 0','*.ue[0].mobility.is3D = true',f'*.ue[0].numApps = {len(flows)}',f'*.edgeNode.numApps = {2*len(flows)}',f'*.missionBackend.numApps = {len(flows)}']
    for i,f in enumerate(flows):
        interval=1.0/f['rate']; ini += [f'*.edgeNode.app[{i}].typename = "CbrReceiver"',f'*.edgeNode.app[{i}].localPort = {6701+i}',f'*.ue[0].app[{i}].typename = "CbrSender"',f'*.ue[0].app[{i}].destAddress = "edgeNode"',f'*.ue[0].app[{i}].destPort = {6701+i}',f'*.ue[0].app[{i}].localPort = {7701+i}',f'*.ue[0].app[{i}].PacketSize = {f["size"]}',f'*.ue[0].app[{i}].sampling_time = {interval:.9f}s','*.ue[0].app[%d].startTime = 0.1s'%i,f'*.ue[0].app[{i}].finishTime = {duration-.1:.3f}s',f'*.edgeNode.app[{len(flows)+i}].typename = "CbrSender"',f'*.edgeNode.app[{len(flows)+i}].destAddress = "missionBackend"',f'*.edgeNode.app[{len(flows)+i}].destPort = {8701+i}',f'*.edgeNode.app[{len(flows)+i}].localPort = {9701+i}',f'*.edgeNode.app[{len(flows)+i}].PacketSize = {f["size"]}',f'*.edgeNode.app[{len(flows)+i}].sampling_time = {interval:.9f}s','*.edgeNode.app[%d].startTime = 0.105s'%(len(flows)+i),f'*.edgeNode.app[{len(flows)+i}].finishTime = {duration-.095:.3f}s',f'*.missionBackend.app[{i}].typename = "CbrReceiver"',f'*.missionBackend.app[{i}].localPort = {8701+i}']
    (inp/'omnetpp.ini').write_text('\n'.join(ini)+'\n')
    cfg=SCENARIOS[scenario]; manifest=dict(organization_id='eirax',fleet_id='fleet_1',drone_id='uav_1',mission_id=f'{RUN.name}_{scenario}',domain='public_safety',mission_type='search_and_rescue',scenario=scenario,attack_type=cfg['attack_type'],attack_stage=cfg['attack_stage'],severity=cfg['severity'],incident_label=cfg['incident_label'],recommended_response=RESPONSES[cfg['attack_type']],seed=seed,rosbag_source=str(RUN/'rosbag_benign'))
    (inp/'manifest.json').write_text(json.dumps(manifest,indent=2)); (inp/'flow_contract.json').write_text(json.dumps(flows,indent=2))
    return root

def run_network(scenario='baseline', seed=0):
    root=prepare_network(scenario,seed); inp=root/'input'; raw=root/'raw'
    script=f'''set -e; source {OMNETPP}/setenv -q; source /home/basudeo/inet/setenv -q; cd {shlex.quote(str(SIMU5G))}; source ./setenv -f; cd {shlex.quote(str(inp))}; simu5g -n {SIMU5G}/simulations:{SIMU5G}/emulation:{SIMU5G}/src:{PROJECT}/UGV_UAV_5G_CoSimulation/sim/simu5g/inet4.5/src:{inp} -u Cmdenv -r 0 --result-dir={raw}; opp_scavetool x {raw}/uas_benign.sca {raw}/uas_benign.vec -o {raw}/network_metrics.csv'''
    result=subprocess.run(['bash','-lc',script],capture_output=True,text=True); (root/'simu5g.log').write_text(result.stdout+'\n'+result.stderr)
    assert result.returncode==0, f'Simu5G failed: {root / "simu5g.log"}'
    assert (raw/'network_metrics.csv').stat().st_size>0
    print('[SIMU5G OK]',scenario,raw/'network_metrics.csv'); return root

BASELINE_NETWORK = run_network('baseline')

[SIMU5G OK] baseline /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/network/baseline/raw/network_metrics.csv


## 6. Python data bridge

The bridge below is the supplied Listing 7 logic completed for measured Simu5G vectors. It creates one row per flow and one-second window. Flags are calculated from measured behavior before labels are attached from the manifest.

In [7]:
import statistics
import pandas as pd

COLUMNS=json.loads(SCHEMA.read_text())['columns']
KNOWN_DESTINATIONS={'10.0.0.2'}
BASELINE_RATE_BY_ACTIVITY={'telemetry':20.0,'command':2.0,'authentication':5.0,'scan':5.0,'api':5.0}

def traffic_burst_flag(row, baseline_pps): return int(row['packets_per_second'] > 3.0*baseline_pps)
def unusual_destination_flag(destination, known_destinations): return int(destination not in known_destinations)
def session_reuse_flag(session_id, source_ip, seen_sessions):
    old=seen_sessions.get(session_id); seen_sessions.setdefault(session_id,source_ip); return int(old is not None and old!=source_ip)
def vector_rows(path):
    csv.field_size_limit(sys.maxsize); result=[]
    with path.open(newline='') as f:
        for r in csv.DictReader(f):
            if r.get('type')=='vector': result.append(r)
    return result
def vector_values(vectors,module,name):
    for r in vectors:
        if r.get('module')==module and r.get('name')==name:
            return [(float(t),float(v)) for t,v in zip(r.get('vectime','').split(),r.get('vecvalue','').split())]
    return []

def assemble(root, window=1.0):
    manifest=json.loads((root/'input/manifest.json').read_text()); flows=json.loads((root/'input/flow_contract.json').read_text()); vectors=vector_rows(root/'raw/network_metrics.csv')
    sinr=vector_values(vectors,'EiraXOneUas.ue[0].cellularNic.nrChannelModel[0]','measuredSinrUl:vector')
    seen_sessions={}; rows=[]; epoch=datetime(2026,1,1,tzinfo=timezone.utc)
    for i,flow in enumerate(flows):
        module=f'EiraXOneUas.missionBackend.app[{i}]'
        sizes=vector_values(vectors,module,'cbrReceivedBytes:vector'); delays=vector_values(vectors,module,'cbrFrameDelay:vector')
        end=max([t for t,_ in sizes],default=0); bins=range(max(1,int(math.ceil(end))))
        for b in bins:
            lo=b*window; hi=(b+1)*window; received=[v for t,v in sizes if lo<=t<hi]; delay=[v*1000 for t,v in delays if lo<=t<hi]; sig=[v for t,v in sinr if lo<=t<hi]
            expected=max(1,round(flow['rate']*window)); count=len(received); pps=count/window; loss=min(1.0,max(0.0,(expected-count)/expected)); jitter=statistics.pstdev(delay) if len(delay)>1 else 0.0
            row=dict(timestamp=(epoch+__import__('datetime').timedelta(seconds=lo)).isoformat(),organization_id=manifest['organization_id'],fleet_id=manifest['fleet_id'],drone_id=manifest['drone_id'],mission_id=manifest['mission_id'],domain=manifest['domain'],mission_type=manifest['mission_type'],source_component=flow['source_component'],destination_component=flow['destination_component'],source_ip=flow['source_ip'],destination_ip=flow['destination_ip'],source_port=flow['source_port'],destination_port=flow['destination_port'],protocol=flow['protocol'],network_slice_id='slice_1',cell_id='gnb_1',session_id=flow['session_id'],packet_count=count,byte_count=int(sum(received)),packets_per_second=round(pps,6),throughput_kbps=round(sum(received)*8/window/1000,6),latency_ms=round(statistics.mean(delay),6) if delay else 0.0,jitter_ms=round(jitter,6),packet_loss_rate=round(loss,6),retransmission_rate=0.0,connection_duration_sec=int(hi),authentication_status=flow['authentication_status'],handover_event=0,handover_count=0,signal_quality_dbm=round(max(-200,min(0,(statistics.mean(sig) if sig else 0)-97)),6),command_channel_activity=int(flow['activity']=='command' and count>0),video_stream_activity=0,telemetry_stream_activity=int(flow['activity']=='telemetry' and count>0),api_request_count=count if flow['activity']=='api' else 0,failed_connection_attempts=count if flow['authentication_status']=='failed' else 0)
            row['unusual_destination_flag']=unusual_destination_flag(row['destination_ip'],KNOWN_DESTINATIONS)
            row['session_reuse_flag']=session_reuse_flag(row['session_id'],row['source_ip'],seen_sessions); row['traffic_burst_flag']=traffic_burst_flag(row,BASELINE_RATE_BY_ACTIVITY.get(flow['activity'],5.0))
            evidence=(row['packet_loss_rate']+row['traffic_burst_flag']+row['unusual_destination_flag']+row['session_reuse_flag']+int(row['authentication_status'] in ('failed','unknown')))/5.0
            row['anomaly_score']=round(min(1.0,evidence),6)
            injected=(manifest['scenario']!='baseline' and flow['name']==manifest['scenario'])
            row.update(attack_type=manifest['attack_type'] if injected else 'none',attack_stage=manifest['attack_stage'] if injected else 'none',severity=manifest['severity'] if injected else 'none',recommended_response=manifest['recommended_response'] if injected else RESPONSES['none'],incident_label=manifest['incident_label'] if injected else 'normal')
            rows.append(row)
    out=root/'data/drone_network_telemetry_cosim.csv'; pd.DataFrame(rows,columns=COLUMNS).to_csv(out,index=False); print('[BRIDGE OK]',len(rows),'rows',out); return out

BASELINE_CSV=assemble(BASELINE_NETWORK)

[BRIDGE OK] 165 rows /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/network/baseline/data/drone_network_telemetry_cosim.csv


## 7. Suspicious and malicious scenarios

Each scenario runs separately against the same measured UAV mobility: one suspicious non-attack case and the six attacks required by the document. This avoids class overlap and preserves scenario provenance.

In [8]:
SCENARIO_OUTPUTS={'baseline':BASELINE_CSV}
for scenario in SCENARIOS:
    if scenario=='baseline': continue
    root=run_network(scenario); SCENARIO_OUTPUTS[scenario]=assemble(root)
scenario_manifest=RUN/'scenario_outputs.json'
scenario_manifest.write_text(json.dumps({k:str(v) for k,v in SCENARIO_OUTPUTS.items()},indent=2))
assert set(SCENARIO_OUTPUTS)==set(SCENARIOS)
print('[SCENARIOS OK] baseline + suspicious + six malicious scenarios')

[SIMU5G OK] suspicious /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/network/suspicious/raw/network_metrics.csv
[BRIDGE OK] 248 rows /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/network/suspicious/data/drone_network_telemetry_cosim.csv
[SIMU5G OK] denial_of_service /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/network/denial_of_service/raw/network_metrics.csv
[BRIDGE OK] 248 rows /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/network/denial_of_service/data/drone_network_telemetry_cosim.csv
[SIMU5G OK] session_hijacking /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/network/session_hijacking/raw/network_metrics.csv
[BRIDGE OK] 248 rows /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/network/session_hijacking/data/drone_network_telemetry_cosim.csv
[SIMU5G OK] identity_theft /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_run

## 8. Concatenate, validate, and evaluate

The final cell concatenates all scenario rows, enforces the authoritative 44-column schema and response mapping, and records accuracy, macro-F1, confusion matrix, false-alarm rate, detection delay, and feature importance. The compact baseline is implemented here because the supplied material did not include a separate baseline notebook.

In [9]:
import ipaddress
import numpy as np

schema=json.loads(SCHEMA.read_text()); frames=[pd.read_csv(p) for p in SCENARIO_OUTPUTS.values()]; final=pd.concat(frames,ignore_index=True)
FINAL_CSV=RUN/'data/drone_network_telemetry_cosim.csv'; FINAL_CSV.parent.mkdir(exist_ok=True); final.to_csv(FINAL_CSV,index=False)

def validate_dataset(frame,schema):
    errors=[]
    if list(frame.columns)!=schema['columns']: errors.append('column names or order do not match schema')
    for col,allowed in schema['categorical_values'].items():
        bad=set(frame[col].dropna().astype(str))-set(allowed)
        if bad: errors.append(f'{col}: invalid {sorted(bad)}')
    for col in schema['integer_nonnegative']:
        v=pd.to_numeric(frame[col],errors='coerce');
        if v.isna().any() or (v<0).any() or ((v%1)!=0).any(): errors.append(f'{col}: must be nonnegative integers')
    for col in schema['number_nonnegative']:
        v=pd.to_numeric(frame[col],errors='coerce');
        if v.isna().any() or (v<0).any(): errors.append(f'{col}: must be nonnegative')
    for col in schema['unit_interval']:
        v=pd.to_numeric(frame[col],errors='coerce');
        if v.isna().any() or not v.between(0,1).all(): errors.append(f'{col}: must be in [0,1]')
    for col in schema['binary']:
        if not set(pd.to_numeric(frame[col],errors='coerce').dropna().unique()).issubset({0,1}): errors.append(f'{col}: must be binary')
    for col in schema['ports']:
        if not pd.to_numeric(frame[col],errors='coerce').between(0,65535).all(): errors.append(f'{col}: invalid port')
    for col in schema['ip_addresses']:
        for value in frame[col].astype(str).unique():
            try: ipaddress.ip_address(value)
            except ValueError: errors.append(f'{col}: invalid IP {value}')
    lo=schema['signal_quality_dbm']['minimum']; hi=schema['signal_quality_dbm']['maximum']
    if not pd.to_numeric(frame.signal_quality_dbm,errors='coerce').between(lo,hi).all(): errors.append('signal_quality_dbm outside range')
    for attack,response in schema['response_by_attack'].items():
        if not (frame.loc[frame.attack_type==attack,'recommended_response']==response).all(): errors.append(f'{attack}: wrong response mapping')
    if errors: raise AssertionError('\n'.join(errors))
    return True

validate_dataset(final,schema); print('[VALIDATION PASSED]',FINAL_CSV,len(final),'rows')
features=['packets_per_second','throughput_kbps','latency_ms','jitter_ms','packet_loss_rate','signal_quality_dbm','failed_connection_attempts','unusual_destination_flag','session_reuse_flag','traffic_burst_flag','anomaly_score']
labels=['normal','suspicious','malicious']; X=final[features].astype(float).to_numpy(); y=final.incident_label.to_numpy(); windows=final.groupby('mission_id').cumcount().to_numpy(); test=(windows%5==0); train=~test
mu=X[train].mean(0); sd=X[train].std(0); sd[sd==0]=1; Z=(X-mu)/sd; centroids={c:Z[train & (y==c)].mean(0) for c in labels}
pred=np.array([min(labels,key=lambda c:float(np.sum((z-centroids[c])**2))) for z in Z])
cm=np.array([[int(np.sum((y[test]==a)&(pred[test]==b))) for b in labels] for a in labels]); accuracy=float(np.trace(cm)/max(1,cm.sum()))
f1=[]
for i in range(len(labels)):
    tp=cm[i,i]; fp=cm[:,i].sum()-tp; fn=cm[i,:].sum()-tp; precision=tp/max(1,tp+fp); recall=tp/max(1,tp+fn); f1.append(2*precision*recall/max(1e-12,precision+recall))
macro_f1=float(np.mean(f1)); normal_i=labels.index('normal'); false_alarm_rate=float((cm[normal_i].sum()-cm[normal_i,normal_i])/max(1,cm[normal_i].sum()))
importance={features[j]:float(np.var([centroids[c][j] for c in labels])) for j in range(len(features))}; total=sum(importance.values()) or 1; importance={k:v/total for k,v in sorted(importance.items(),key=lambda x:x[1],reverse=True)}
delays={}
for mission,g in final[final.incident_label!='normal'].groupby('mission_id'):
    detected=g[g.anomaly_score>=0.2]; delays[mission]=None if detected.empty else int(detected.index[0]-g.index[0])
metrics=dict(split='deterministic temporal holdout: every fifth window per scenario',accuracy=accuracy,macro_f1=macro_f1,false_alarm_rate=false_alarm_rate,confusion_matrix_labels=labels,confusion_matrix=cm.tolist(),detection_delay_windows=delays,feature_importance=importance)
(RUN/'data/baseline_metrics.json').write_text(json.dumps(metrics,indent=2)); print(json.dumps(metrics,indent=2))

[VALIDATION PASSED] /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/data/drone_network_telemetry_cosim.csv 1901 rows
{
  "split": "deterministic temporal holdout: every fifth window per scenario",
  "accuracy": 0.95822454308094,
  "macro_f1": 0.9618272841051314,
  "false_alarm_rate": 0.0,
  "confusion_matrix_labels": [
    "normal",
    "suspicious",
    "malicious"
  ],
  "confusion_matrix": [
    [
      264,
      0,
      0
    ],
    [
      0,
      17,
      0
    ],
    [
      16,
      0,
      86
    ]
  ],
  "detection_delay_windows": {
    "run_20260805_210640_control_protocol_attack": 0,
    "run_20260805_210640_denial_of_service": 0,
    "run_20260805_210640_identity_theft": 0,
    "run_20260805_210640_lateral_movement": 0,
    "run_20260805_210640_network_application_attack": 0,
    "run_20260805_210640_session_hijacking": 0,
    "run_20260805_210640_suspicious": 0
  },
  "feature_importance": {
    "jitter_ms": 0.4026597849059926,
    "traffic

## Safe shutdown

In [10]:
for name in ('rosbag','mavros','px4','gazebo'):
    stop_process(PROCESSES.get(name))
for handle in LOG_HANDLES:
    if not handle.closed: handle.close()
print('[STOPPED] Notebook processes closed safely')

[STOPPED] Notebook processes closed safely


## Document 1 completion — run only these new cells

The completed and saved run above is unchanged. Run only the new cells below, in order. They recover the completed run and finish its Document 1 evidence without repeating the Gazebo flight.

The security conditions are trace-driven network runs. The same measured UAV flight is deliberately reused so the network condition is the controlled variable.

In [13]:
# Recover the completed run without rerunning Gazebo, PX4, or the mission.
from pathlib import Path
import csv, json, os, shlex, shutil, subprocess, sys
import pandas as pd
HERE=Path.cwd().resolve(); assert HERE.name=='UAV_5GSim', f'Open notebook from UAV_5GSim, not {HERE}'
SCHEMA=HERE/'schema_reference.json'
runs=sorted((HERE/'notebook_runs').glob('run_*')); assert runs, 'No completed run found'
RUN=Path(RUN) if 'RUN' in globals() and Path(RUN).is_dir() else runs[-1]
DOC1=RUN/'document1_completion'; DOC1.mkdir(exist_ok=True)
FINAL_CSV=RUN/'data/drone_network_telemetry_cosim.csv'
manifest_path=RUN/'scenario_outputs.json'
assert FINAL_CSV.is_file() and manifest_path.is_file(), 'Completed dataset/scenario manifest missing'
SCENARIO_OUTPUTS={k:Path(v) for k,v in json.loads(manifest_path.read_text()).items()}
EXPECTED={'baseline','suspicious','denial_of_service','session_hijacking','identity_theft','control_protocol_attack','lateral_movement','network_application_attack'}
assert set(SCENARIO_OUTPUTS)==EXPECTED
print('[RECOVERED]',RUN)
print('[UNCHANGED] Completed cells above will not be rerun')

[RECOVERED] /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640
[UNCHANGED] Completed cells above will not be rerun


### ROS measurements, ROS pose export, and shared-clock join

This reads the saved MCAP bag directly to measure message frequency, serialized size, and bandwidth. It also exports the ROS pose and audits the timestamp join between every dataset flow/window and the measured UAV pose.

In [14]:
ROS_BAG=RUN/'rosbag_benign'; assert (ROS_BAG/'metadata.yaml').is_file()
extractor=r"""
from pathlib import Path
import csv,sys,rosbag2_py
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message
bag,out=Path(sys.argv[1]),Path(sys.argv[2]); out.mkdir(parents=True,exist_ok=True)
r=rosbag2_py.SequentialReader(); r.open(rosbag2_py.StorageOptions(uri=str(bag),storage_id='mcap'),rosbag2_py.ConverterOptions('',''))
types={x.name:x.type for x in r.get_all_topics_and_types()}; count={k:0 for k in types}; size={k:0 for k in types}; first={}; last={}; pose=[]
pose_type=get_message(types['/mavros/local_position/pose'])
while r.has_next():
 topic,data,t=r.read_next(); count[topic]+=1; size[topic]+=len(data); first.setdefault(topic,t); last[topic]=t
 if topic=='/mavros/local_position/pose':
  p=deserialize_message(data,pose_type).pose.position; pose.append((t,p.x,p.y,p.z))
with (out/'topic_measurements.csv').open('w',newline='') as f:
 w=csv.writer(f); w.writerow(['topic','message_count','duration_sec','frequency_hz','total_serialized_bytes','average_serialized_message_bytes','bandwidth_kbps'])
 for topic in sorted(types):
  duration=max(1e-9,(last[topic]-first[topic])/1e9); n=count[topic]
  w.writerow([topic,n,duration,n/duration,size[topic],size[topic]/max(1,n),size[topic]*8/duration/1000])
assert pose; t0,x0,y0,z0=pose[0]
with (out/'ros_pose_trace.csv').open('w',newline='') as f:
 w=csv.writer(f); w.writerow(['elapsed_sec','east_m','north_m','up_m'])
 for t,x,y,z in pose:w.writerow([(t-t0)/1e9,x-x0,y-y0,z-z0])
with (out/'simu5g_pose_trace_from_ros.txt').open('w') as f:
 f.write('time_sec x_east_m y_north_m z_up_m\n')
 for t,x,y,z in pose:f.write(f'{(t-t0)/1e9} {x-x0} {y-y0} {z-z0}\n')
"""
cmd=f'source /opt/ros/jazzy/setup.bash && exec /usr/bin/python3 -c {shlex.quote(extractor)} {shlex.quote(str(ROS_BAG))} {shlex.quote(str(DOC1))}'
result=subprocess.run(['bash','-lc',cmd],capture_output=True,text=True,timeout=180); assert result.returncode==0,result.stdout+'\n'+result.stderr
measurements=pd.read_csv(DOC1/'topic_measurements.csv'); required={'/mavros/local_position/pose','/mavros/state','/mavros/global_position/raw/fix'}
assert required.issubset(set(measurements.topic)) and (measurements.set_index('topic').loc[list(required),'message_count']>0).all()
ros_pose=pd.read_csv(DOC1/'ros_pose_trace.csv'); final=pd.read_csv(FINAL_CSV); final['_time']=pd.to_datetime(final.timestamp,utc=True)
joined=[]
for mission,g in final.groupby('mission_id',sort=False):
 g=g.copy(); g['elapsed_sec']=(g['_time']-g['_time'].min()).dt.total_seconds()
 j=pd.merge_asof(g.sort_values('elapsed_sec'),ros_pose.sort_values('elapsed_sec'),on='elapsed_sec',direction='nearest',tolerance=1.0)
 j.insert(0,'joined_mission_id',mission); joined.append(j[['joined_mission_id','timestamp','elapsed_sec','east_m','north_m','up_m']])
clock_audit=pd.concat(joined,ignore_index=True); clock_audit['ros_pose_matched']=clock_audit.east_m.notna().astype(int); clock_audit.to_csv(DOC1/'shared_clock_join_audit.csv',index=False)
coverage=float(clock_audit.ros_pose_matched.mean()); assert coverage==1.0
final.drop(columns=['_time'],inplace=True)
print(measurements.to_string(index=False)); print(f'[POSE EXPORT PASSED] {len(ros_pose)} samples'); print(f'[SHARED CLOCK JOIN PASSED] {coverage:.0%} of {len(clock_audit)} rows')

                          topic  message_count  duration_sec  frequency_hz  total_serialized_bytes  average_serialized_message_bytes  bandwidth_kbps
/mavros/global_position/raw/fix            516    103.715774      4.975135                   70176                        136.000000        5.412947
    /mavros/local_position/pose           5125    103.167930     49.676290                  389500                         76.000000       30.203184
                  /mavros/state         268771    528.210100    508.833512                11825556                         43.998631      179.103823
[POSE EXPORT PASSED] 5125 samples
[SHARED CLOCK JOIN PASSED] 100% of 1901 rows


### Scenario ROS bags and measured injection evidence

Each isolated network scenario is associated with the measured flight bag using storage-efficient hard links. The scenario manifest states this controlled trace-reuse strategy. Measured Simu5G output—not the label—is then checked for the expected behavioural effect.

In [15]:
INJECTION={
'baseline':'none','suspicious':'ambiguous authentication retry; no malicious attack','denial_of_service':'INET high-rate flood flow','session_hijacking':'rogue command-session source','identity_theft':'rogue identity source with failed authentication','control_protocol_attack':'extra MAVLink-like command flow','lateral_movement':'INET internal-destination scanner flow','network_application_attack':'mission-backend API flood flow'}
bag_root=DOC1/'scenario_rosbags'; bag_root.mkdir(exist_ok=True)
for scenario,data_csv in SCENARIO_OUTPUTS.items():
 target=bag_root/scenario
 if not target.exists():
  try: shutil.copytree(ROS_BAG,target,copy_function=os.link)
  except OSError: shutil.copytree(ROS_BAG,target,copy_function=shutil.copy2)
 (target/'scenario_manifest.json').write_text(json.dumps({'scenario':scenario,'injection_point':INJECTION[scenario],'rosbag_strategy':'same measured flight used as controlled trace-driven mobility input','source_rosbag':str(ROS_BAG),'network_dataset':str(data_csv)},indent=2))
 info=subprocess.run(['bash','-lc',f'source /opt/ros/jazzy/setup.bash && ros2 bag info {shlex.quote(str(target))}'],capture_output=True,text=True,timeout=30)
 assert info.returncode==0 and 'Messages:' in info.stdout
evidence=[]
for scenario,path in SCENARIO_OUTPUTS.items():
 f=pd.read_csv(path); x=f[f.attack_type!='none']
 if scenario=='baseline': ok=len(x)==0 and set(f.incident_label)=={'normal'}; detail='normal measured flows only'
 elif scenario=='suspicious': ok=len(x)>0 and set(x.incident_label)=={'suspicious'} and (x.authentication_status=='unknown').all() and x.anomaly_score.gt(0).any(); detail='ambiguous authentication behaviour'
 elif scenario=='denial_of_service': ok=x.traffic_burst_flag.eq(1).any() and x.packets_per_second.mean()>100; detail='packet-rate and burst increase'
 elif scenario=='session_hijacking': ok=x.session_reuse_flag.eq(1).any() and x.command_channel_activity.eq(1).any(); detail='session reuse and command activity'
 elif scenario=='identity_theft': ok=x.failed_connection_attempts.gt(0).any() and (x.authentication_status=='failed').all(); detail='failed rogue-identity authentication'
 elif scenario=='control_protocol_attack': ok=x.command_channel_activity.eq(1).any() and x.traffic_burst_flag.eq(1).any(); detail='command-channel spike'
 elif scenario=='lateral_movement': ok=x.unusual_destination_flag.eq(1).any(); detail='unusual internal destination'
 else: ok=x.api_request_count.gt(0).any() and x.traffic_burst_flag.eq(1).any(); detail='API request and burst increase'
 evidence.append({'scenario':scenario,'injection_point':INJECTION[scenario],'measured_evidence':detail,'passed':bool(ok),'rows':len(f)})
evidence=pd.DataFrame(evidence); evidence.to_csv(DOC1/'scenario_evidence.csv',index=False); assert evidence.passed.all()
print(evidence.to_string(index=False)); print('[SCENARIO BAGS PASSED] benign, suspicious, and six malicious packages'); print('[INJECTION EVIDENCE PASSED]')

                  scenario                                     injection_point                    measured_evidence  passed  rows
                  baseline                                                none           normal measured flows only    True   165
                suspicious ambiguous authentication retry; no malicious attack   ambiguous authentication behaviour    True   248
         denial_of_service                           INET high-rate flood flow       packet-rate and burst increase    True   248
         session_hijacking                        rogue command-session source   session reuse and command activity    True   248
            identity_theft    rogue identity source with failed authentication failed rogue-identity authentication    True   248
   control_protocol_attack                     extra MAVLink-like command flow                command-channel spike    True   248
          lateral_movement              INET internal-destination scanner flow         unu

### Behavioural-flag audit, standalone validation, and final Document 1 checklist

The behavioural fields are independently recomputed without consulting the attack labels. A standalone validator is exported from this self-contained notebook and executed against the final 44-column CSV.

In [16]:
# Recompute behavioural flags independently of attack_type and incident_label.
checks=[]
for mission,g in final.groupby('mission_id',sort=False):
 seen={}
 for _,r in g.iterrows():
  base=20.0 if r.telemetry_stream_activity==1 else (2.0 if r.command_channel_activity==1 else 5.0)
  burst=int(r.packets_per_second>3*base); unusual=int(r.destination_ip!='10.0.0.2'); old=seen.get(r.session_id); seen.setdefault(r.session_id,r.source_ip); reuse=int(old is not None and old!=r.source_ip)
  score=round(min(1.0,(r.packet_loss_rate+burst+unusual+reuse+int(r.authentication_status in ('failed','unknown')))/5),6)
  checks.append((burst==r.traffic_burst_flag,unusual==r.unusual_destination_flag,reuse==r.session_reuse_flag,abs(score-r.anomaly_score)<1e-6))
flag_audit=pd.DataFrame(checks,columns=['traffic_burst_matches','unusual_destination_matches','session_reuse_matches','anomaly_score_matches']); flag_audit.to_csv(DOC1/'behavioural_flag_audit.csv',index=False); assert flag_audit.all().all()

VALIDATOR=HERE/'validate_dataset.py'
validator_source=r"""import ipaddress,json,sys
import pandas as pd
df=pd.read_csv(sys.argv[1]); s=json.load(open(sys.argv[2])); e=[]
if list(df.columns)!=s['columns']:e.append('column names/order mismatch')
for c,a in s['categorical_values'].items():
 b=set(df[c].dropna().astype(str))-set(a)
 if b:e.append(f'{c}: invalid {sorted(b)}')
for c in s['integer_nonnegative']:
 v=pd.to_numeric(df[c],errors='coerce')
 if v.isna().any() or (v<0).any() or ((v%1)!=0).any():e.append(f'{c}: invalid integer')
for c in s['number_nonnegative']:
 v=pd.to_numeric(df[c],errors='coerce')
 if v.isna().any() or (v<0).any():e.append(f'{c}: invalid number')
for c in s['unit_interval']:
 v=pd.to_numeric(df[c],errors='coerce')
 if v.isna().any() or not v.between(0,1).all():e.append(f'{c}: outside [0,1]')
for c in s['binary']:
 v=pd.to_numeric(df[c],errors='coerce')
 if v.isna().any() or not set(v.unique()).issubset({0,1}):e.append(f'{c}: not binary')
for c in s['ports']:
 if not pd.to_numeric(df[c],errors='coerce').between(0,65535).all():e.append(f'{c}: bad port')
for c in s['ip_addresses']:
 for x in df[c].astype(str).unique():
  try:ipaddress.ip_address(x)
  except ValueError:e.append(f'{c}: bad IP {x}')
q=s['signal_quality_dbm']
if not pd.to_numeric(df.signal_quality_dbm,errors='coerce').between(q['minimum'],q['maximum']).all():e.append('signal quality out of range')
for a,r in s['response_by_attack'].items():
 if not (df.loc[df.attack_type==a,'recommended_response']==r).all():e.append(f'{a}: response mismatch')
if e:print('RESULT: FAILED\n'+'\n'.join(e));raise SystemExit(1)
print(f'RESULT: PASSED ({len(df)} rows, {len(df.columns)} columns)')
"""
VALIDATOR.write_text(validator_source)
validation=subprocess.run([sys.executable,str(VALIDATOR),str(FINAL_CSV),str(SCHEMA)],capture_output=True,text=True); print(validation.stdout,end=''); assert validation.returncode==0,validation.stderr
metrics_path=RUN/'data/baseline_metrics.json'; metrics=json.loads(metrics_path.read_text())
saved_nb=json.loads((HERE/'UAV_5GSim.ipynb').read_text()); saved_text='\n'.join(''.join(o.get('text',[])) for c in saved_nb['cells'] for o in c.get('outputs',[]) if o.get('output_type')=='stream')
deliverables={
'Gazebo/PX4 waypoint mission landed':'[LANDED] UAV landed and disarmed at the goal' in saved_text,
'ROS bags for benign, suspicious, malicious':all((bag_root/s/'metadata.yaml').is_file() for s in EXPECTED),
'ROS topic rate and size measurements':(DOC1/'topic_measurements.csv').stat().st_size>0,
'Pose trace exported':(DOC1/'simu5g_pose_trace_from_ros.txt').stat().st_size>0,
'Pose used for Simu5G mobility':all((p.parent.parent/'input/uav.movements').stat().st_size>0 for p in SCENARIO_OUTPUTS.values()),
'Simu5G metrics CSV files':all((p.parent.parent/'raw/network_metrics.csv').stat().st_size>0 for p in SCENARIO_OUTPUTS.values()),
'One row per flow/window':len(final)>0 and not final[['mission_id','timestamp','source_port','destination_port']].duplicated().any(),
'ROS/network shared-clock join':coverage==1.0,
'Behavioural flags derived from telemetry':flag_audit.all().all(),
'Exact 44-column schema':len(final.columns)==44,
'Standalone validator passed':validation.returncode==0,
'Baseline metrics reported':all(k in metrics for k in ('accuracy','macro_f1','confusion_matrix','feature_importance'))}
deliverables={name:bool(ok) for name,ok in deliverables.items()}
for name,ok in deliverables.items():print('[PASS]' if ok else '[FAIL]',name)
assert all(deliverables.values())
summary={'run':str(RUN),'dataset':str(FINAL_CSV),'rows':len(final),'columns':len(final.columns),'deliverables':deliverables,'metrics':metrics}; (DOC1/'document1_completion_summary.json').write_text(json.dumps(summary,indent=2))
seen=set(); used=0
for x in RUN.rglob('*'):
 if x.is_file():
  st=x.stat(); key=(st.st_dev,st.st_ino)
  if key not in seen: seen.add(key); used+=st.st_size
free=shutil.disk_usage(HERE).free
print(f'[STORAGE] Run uses {used/1024**2:.1f} MiB of unique files; free {free/1024**3:.1f} GiB'); print('[DOCUMENT 1 COMPLETE]',DOC1/'document1_completion_summary.json')

RESULT: PASSED (1901 rows, 44 columns)
[PASS] Gazebo/PX4 waypoint mission landed
[PASS] ROS bags for benign, suspicious, malicious
[PASS] ROS topic rate and size measurements
[PASS] Pose trace exported
[PASS] Pose used for Simu5G mobility
[PASS] Simu5G metrics CSV files
[PASS] One row per flow/window
[PASS] ROS/network shared-clock join
[PASS] Behavioural flags derived from telemetry
[PASS] Exact 44-column schema
[PASS] Standalone validator passed
[PASS] Baseline metrics reported
[STORAGE] Run uses 313.0 MiB of unique files; free 234.6 GiB
[DOCUMENT 1 COMPLETE] /home/basudeo/Documents/EiraX/UAV_5GSim/notebook_runs/run_20260805_210640/document1_completion/document1_completion_summary.json
